# Control Database for Missing Data and Values

In [3]:
import numpy as np
import pandas as pd
from pathlib import Path

DB_PATH = Path(r"C:\Users\benel\Coding\Python\Thesis\01_database\database.parquet")
df = pd.read_parquet(DB_PATH).sort_index()

# ---- normalise columns: level 0 = field (Open/High/Low/Close/Volume), level 1 = ticker
fields       = df.columns.get_level_values(0)
all_fields   = list(dict.fromkeys(fields))
uniq_tickers = list(dict.fromkeys(df.columns.get_level_values(1)))
price_fields = [f for f in ("Open", "High", "Low", "Close") if f in set(fields)]

print("=" * 72)
print(f"DATABASE QUALITY REPORT  —  {DB_PATH.name}")
print("=" * 72)
print(f"Rows (dates)   : {len(df):,}")
print(f"Columns        : {df.shape[1]:,}  ({len(all_fields)} fields x {len(uniq_tickers)} tickers)")
print(f"Date range     : {df.index.min():%Y-%m-%d}  ->  {df.index.max():%Y-%m-%d}")
print(f"Fields         : {all_fields}")

# ---- 1. INDEX INTEGRITY -------------------------------------------------------
print("\n--- INDEX INTEGRITY ---")
print(f"Monotonic increasing : {df.index.is_monotonic_increasing}")
print(f"Duplicate dates      : {int(df.index.duplicated().sum())}")
bdays = pd.bdate_range(df.index.min(), df.index.max())
print(f"Business days in span : {len(bdays):,}")
print(f"Rows present          : {len(df):,}  ({len(df)/len(bdays):.1%} of business days; "
      f"gap to 100% = market holidays, expected)")
gaps = df.index.to_series().diff().dt.days
big = gaps[gaps > 5]
print(f"Largest date gap      : {int(gaps.max())} days ending {gaps.idxmax():%Y-%m-%d}")
print("Date gaps > 5 calendar days:")
print("\n".join(f"   {d:%Y-%m-%d}  (+{int(g)}d)" for d, g in big.items()) or "   none")

# ---- 2. PER-TICKER COVERAGE / FULLNESS (based on Close) ----------------------
print("\n--- PER-TICKER COVERAGE (Close) ---")
ref = "Close" if "Close" in all_fields else all_fields[0]
rows, n = [], len(df)
for tk in uniq_tickers:
    s = df[(ref, tk)]
    fv, lv = s.first_valid_index(), s.last_valid_index()
    if fv is None:
        rows.append(dict(ticker=tk, first=None, last=None, obs=0, span=0,
                         internal_na=n, lead_na=n, pct_full=0.0))
        continue
    win = s.loc[fv:lv]
    obs = int(win.notna().sum())
    rows.append(dict(
        ticker=tk, first=fv.date(), last=lv.date(),
        obs=obs, span=len(win),
        internal_na=len(win) - obs,               # NaNs mid-history  -> real problem
        lead_na=int(s.loc[:fv].isna().sum()) - 0,  # NaNs before listing -> expected (late IPO)
        pct_full=round(obs / len(win), 4),
    ))
cov = pd.DataFrame(rows).set_index("ticker").sort_values("pct_full")
with pd.option_context("display.max_rows", None, "display.width", 160):
    print(cov)
print(f"\nTickers with internal gaps (missing mid-history) : {int((cov.internal_na > 0).sum())}")
print(f"Tickers not up to date (last < {df.index.max():%Y-%m-%d}) : "
      f"{int((pd.to_datetime(cov['last']) < df.index.max()).sum())}")

# ---- 3. NaN MATRIX (field x ticker), only where present ----------------------
print("\n--- NaN COUNT (field x ticker, non-zero only) ---")
na = df.isna().sum()
na = na[na > 0].unstack(0).fillna(0).astype(int)
print(na if len(na) else "   no NaNs anywhere")

# ---- 4. VALUE SANITY -------------------------------------------------------- 
print("\n--- VALUE SANITY ---")
for f in price_fields:
    print(f"{f:6s}: non-positive values = {int((df[f] <= 0).sum().sum())}")
if "Volume" in all_fields:
    vol = df["Volume"]
    print(f"Volume: negative = {int((vol < 0).sum().sum())}, "
          f"zero = {int((vol == 0).sum().sum())}  (index tickers like ^VIX legitimately have 0)")
if {"Open", "High", "Low", "Close"}.issubset(all_fields):
    o, h, l, c = df["Open"], df["High"], df["Low"], df["Close"]
    print(f"OHLC: High<Low = {int((h < l).sum().sum())}, "
          f"High<max(O,C) = {int((h < o).sum().sum() + (h < c).sum().sum())}, "
          f"Low>min(O,C) = {int((l > o).sum().sum() + (l > c).sum().sum())}")

# ---- 5. OUTLIERS & STALE PRICES (Close) ------------------------------------- 
print("\n--- RETURN / STALENESS CHECKS (Adj Close) ---")
close = df["Adj Close"]
ext = (close.pct_change().abs() > 0.5).sum()
ext = ext[ext > 0].sort_values(ascending=False)
print("Daily |return| > 50% (possible bad tick or unadjusted split):")
print(ext.to_string() if len(ext) else "   none")
flat = ((close == close.shift()) & close.notna()).sum()
flat = flat[flat > 0].sort_values(ascending=False)
print("\nDays with Adj Close unchanged vs prior day (top 10):")
print(flat.head(10).to_string() if len(flat) else "   none")

# ---- 6. FULLNESS BY YEAR (fraction of trading days each ticker has data) ---- 
print("\n--- YEARLY FULLNESS (mean over tickers of 'has Close') ---")
yearly = close.notna().groupby(close.index.year).mean().mean(axis=1).round(3)
print(yearly.to_string())


DATABASE QUALITY REPORT  —  database.parquet
Rows (dates)   : 9,066
Columns        : 360  (8 fields x 45 tickers)
Date range     : 1990-01-02  ->  2025-12-30
Fields         : ['Open', 'High', 'Low', 'Close', 'Adj Close', 'Volume', 'Dividends', 'Stock Splits']

--- INDEX INTEGRITY ---
Monotonic increasing : True
Duplicate dates      : 0
Business days in span : 9,391
Rows present          : 9,066  (96.5% of business days; gap to 100% = market holidays, expected)
Largest date gap      : 7 days ending 2001-09-17
Date gaps > 5 calendar days:
   2001-09-17  (+7d)

--- PER-TICKER COVERAGE (Close) ---
             first        last   obs  span  internal_na  lead_na  pct_full
ticker                                                                    
AAPL    1990-01-02  2025-12-30  9066  9066            0        0       1.0
AIG     1990-01-02  2025-12-30  9066  9066            0        0       1.0
AMGN    1990-01-02  2025-12-30  9066  9066            0        0       1.0
AMZN    1997-05-15  2025